# Train pipeline (U-Net / ResNet-UNet)

## Setup environment

### Colab

In [ ]:
!git clone https://github.com/trxxnk/text-image-alignment.git

In [ ]:
import os
os.chdir("/content/text-image-alignment/")
print(f"Working directory: {os.getcwd()}")

In [ ]:
!git checkout dev

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!chmod +x src/scripts/setup_colab.sh
!src/scripts/setup_colab.sh

### Local

In [1]:
import os, sys
from pathlib import Path

os.chdir(os.path.dirname(sys.prefix))
_repo = Path.cwd().resolve()
os.environ.setdefault("TORCH_HOME", str(_repo / ".cache" / "torch"))
print(f"Working directory: {os.getcwd()}")
print(f"TORCH_HOME={os.environ['TORCH_HOME']}")

Working directory: /home/trxxnk/mycode/diplom
TORCH_HOME=/home/trxxnk/mycode/diplom/.cache/torch


## Import libs

In [2]:
import torch
import mlflow
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
from src.tps_dewarp.training import (
    Trainer,
    load_train_config,
    build_tps_dataloaders,
    build_model,
)

## Setup torch, dagshub, mlflow

In [4]:
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

Accessing as trxxnk

Initialized MLflow to track repo "trxxnk/text-image-alignment"

Repository trxxnk/text-image-alignment initialized!

## Config

In [6]:
# Local:
CONFIG_PATH = "configs/train_unet_local.yaml"
# CONFIG_PATH = "configs/train_resunet_local.yaml"
# Colab:
# CONFIG_PATH = "configs/train_unet.yaml"
# CONFIG_PATH = "configs/train_resunet.yaml"

cfg = load_train_config(CONFIG_PATH)
print("model:", cfg.model_name, "| dataset:", cfg.dataset_dir)
print("epochs:", cfg.epochs, "| batch:", cfg.batch_size, "| amp:", cfg.use_amp)

model: UNetTPS | dataset: data/generated/test2
epochs: 2 | batch: 8 | amp: False


## Dataset

In [7]:
train_loader, val_loader, test_loader = build_tps_dataloaders(cfg, device)
len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

(240, 30, 30)

In [8]:
# Проверка одного элемента train
img, delta, difficulty = train_loader.dataset[0]
print(img.shape)
print(delta.shape)
print(difficulty)

torch.Size([1, 256, 256])
torch.Size([81, 2])
hard


In [9]:
from collections import Counter

subset = train_loader.dataset
diffs = [subset.dataset.samples[i]["difficulty"] for i in subset.indices]
counter = Counter(diffs)
total = sum(counter.values())
print(counter, "total=", total)

Counter({'easy': 76, 'identity': 62, 'hard': 52, 'medium': 50}) total= 240


In [10]:
x, y, d = next(iter(train_loader))
print(x.shape)  # (B, 1, H, W)
print(y.shape)  # (B, 162)
print(len(d))

torch.Size([8, 1, 256, 256])
torch.Size([8, 81, 2])
8


## Model

In [11]:
model = build_model(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{cfg.model_name}: {n_params/1e6:.2f}M параметров")

x = train_loader.dataset[1][0].unsqueeze(0).to(device)
out = model(x)
print("out:", tuple(out.shape), "| target:", tuple(train_loader.dataset[1][1].shape))

UNetTPS: 1.95M параметров
out: (1, 162) | target: (81, 2)


## Train Loop

In [12]:
RUN_NAME = f"UNet_Local_01"
mlflow.set_experiment(cfg.experiment_name)
mlflow.start_run(run_name=RUN_NAME)

2026/06/01 22:16:26 INFO mlflow.tracking.fluent: Experiment with name 'TPS_Dewarp_local' does not exist. Creating a new experiment.


<ActiveRun: >

In [13]:
# Loss, optimizer, scheduler
trainer = Trainer(cfg, model, train_loader, val_loader, test_loader, device)

In [14]:
trainer.fit(resume_from=None)

Epoch 1/2 | train_loss=0.0019 | train_l2_px=7.80 | val_loss=0.0008 | val_l2_px=5.38 | p95=9.80 | gn=0.058 | lr=5.00e-04 | 2.2 samp/s


Epoch 2/2 | train_loss=0.0003 | train_l2_px=5.33 | val_loss=0.0003 | val_l2_px=4.75 | p95=9.51 | gn=0.024 | lr=0.00e+00 | 2.2 samp/s


In [15]:
# Принудительно завершить mlflow run по run_id
run_id = "62d1e54bca0e4ed49f5d6aec8fb3793f"
mlflow.tracking.MlflowClient().set_terminated(run_id)

🏃 View run UNet_Local_01 at: https://dagshub.com/trxxnk/text-image-alignment.mlflow/#/experiments/1/runs/62d1e54bca0e4ed49f5d6aec8fb3793f
🧪 View experiment at: https://dagshub.com/trxxnk/text-image-alignment.mlflow/#/experiments/1
